In [1]:
import sys
sys.path.append('..')

import requests
import pandas as pd
from src.config import CITIES, RAW_BMKG_DIR, INTERIM_DIR

print('Setup OK')

Setup OK


In [2]:
def fetch_nasa_power(lat: float, lon: float,
                     start: str, end: str) -> pd.DataFrame:
    """
    Fetch daily weather from NASA POWER (ERA5 reanalysis).
    Free, no API key. start/end format: 'YYYY-MM-DD'.
    """
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters" : "T2M_MAX,T2M_MIN,T2M,PRECTOTCORR,WS10M_MAX,WD10M,RH2M",
        "community"  : "RE",
        "longitude"  : lon,
        "latitude"   : lat,
        "start"      : start.replace("-", ""),
        "end"        : end.replace("-", ""),
        "format"     : "JSON",
    }
    r = requests.get(url, params=params, timeout=90)
    r.raise_for_status()

    params_data = r.json()["properties"]["parameter"]
    df = pd.DataFrame(params_data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "date"
    df = df.reset_index()
    return df


# Fetch untuk Jakarta
jakarta = CITIES["jakarta"]
print(f"Fetching weather for Jakarta ({jakarta['lat']}, {jakarta['lon']})...")

weather_df = fetch_nasa_power(
    lat   = jakarta["lat"],
    lon   = jakarta["lon"],
    start = "2019-01-01",
    end   = "2023-12-31",
)

print(f"Records: {len(weather_df):,}")
print(f"Date range: {weather_df['date'].min()} to {weather_df['date'].max()}")
weather_df.head()

Fetching weather for Jakarta (-6.2088, 106.8456)...


Records: 1,826
Date range: 2019-01-01 00:00:00 to 2023-12-31 00:00:00


,date,T2M_MAX,T2M_MIN,T2M,PRECTOTCORR,WS10M_MAX,WD10M,RH2M
0,2019-01-01,28.45,25.49,26.93,18.66,6.01,258.4,86.39
1,2019-01-02,30.06,25.21,27.25,24.26,7.08,255.7,83.16
2,2019-01-03,30.55,25.28,27.56,5.69,6.36,242.6,80.62
3,2019-01-04,31.21,25.05,27.83,0.41,4.82,242.6,77.58
4,2019-01-05,30.91,25.18,27.86,1.87,3.96,260.0,76.10


In [3]:
weather_df = weather_df.rename(columns={
    "T2M_MAX"     : "temp_max",
    "T2M_MIN"     : "temp_min",
    "T2M"         : "temp_mean",
    "PRECTOTCORR" : "precipitation",
    "WS10M_MAX"   : "windspeed_max",
    "WD10M"       : "wind_direction",
    "RH2M"        : "humidity_mean",
})

# Add rain flag
weather_df["is_rainy"] = weather_df["precipitation"] > 1.0

print(f"Columns: {weather_df.columns.tolist()}")
print(f"\nMissing values:\n{weather_df.isnull().sum()}")
print(f"\nStats:\n{weather_df.describe().round(2)}")

Columns: ['date', 'temp_max', 'temp_min', 'temp_mean', 'precipitation', 'windspeed_max', 'wind_direction', 'humidity_mean', 'is_rainy']

Missing values:
date              0
temp_max          0
temp_min          0
temp_mean         0
precipitation     0
windspeed_max     0
wind_direction    0
humidity_mean     0
is_rainy          0
dtype: int64

Stats:
                      date  temp_max  temp_min  temp_mean  precipitation  \
count                 1826   1826.00   1826.00    1826.00        1826.00   
mean   2021-07-01 12:00:00     29.70     25.74      27.54           6.30   
min    2019-01-01 00:00:00     26.42     22.15      25.44           0.00   
25%    2020-04-01 06:00:00     29.02     25.28      27.06           1.14   
50%    2021-07-01 12:00:00     29.69     25.79      27.50           3.64   
75%    2022-09-30 18:00:00     30.27     26.28      27.97           7.86   
max    2023-12-31 00:00:00     34.16     28.09      30.04         104.93   
std                    NaN      1.15  

In [4]:
# Load interim AQ data
aq_df = pd.read_parquet(INTERIM_DIR / "jakarta_central_interim.parquet")
aq_df = aq_df.reset_index()

# Merge on date
weather_df["date"] = pd.to_datetime(weather_df["date"])
aq_df["date"] = pd.to_datetime(aq_df["date"])

master_df = aq_df.merge(weather_df, on="date", how="left")

print(f"AQ records: {len(aq_df):,}")
print(f"Weather records: {len(weather_df):,}")
print(f"Master records: {len(master_df):,}")
print(f"\nWeather merge check (NaN after merge):")
print(master_df[["temp_mean", "precipitation", "windspeed_max"]].isnull().sum())

AQ records: 1,826
Weather records: 1,826
Master records: 1,826

Weather merge check (NaN after merge):
temp_mean        0
precipitation    0
windspeed_max    0
dtype: int64


In [5]:
from src.config import PROCESSED_DIR

output_path = PROCESSED_DIR / "jakarta_master.parquet"
master_df.to_parquet(output_path, index=False)

print(f"Master dataset saved: {output_path}")
print(f"Shape: {master_df.shape}")
print(f"\nFinal columns:")
print(master_df.columns.tolist())
print(f"\nSample:")
master_df[["date", "pm25_mean", "temp_mean",
           "precipitation", "windspeed_max",
           "is_psbb", "season"]].head(10)

Master dataset saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\data\processed\jakarta_master.parquet
Shape: (1826, 27)

Final columns:
['date', 'datetime_utc', 'station_name', 'location_id', 'pm25_mean', 'pm25_median', 'pm25_max', 'pm25_min', 'pm25_p02', 'pm25_p98', 'coverage_pct', 'obs_count', 'year', 'month', 'dayofweek', 'quarter', 'season', 'is_psbb', 'psbb_phase', 'temp_max', 'temp_min', 'temp_mean', 'precipitation', 'windspeed_max', 'wind_direction', 'humidity_mean', 'is_rainy']

Sample:


,date,pm25_mean,temp_mean,precipitation,windspeed_max,is_psbb,season
0,2019-01-01,NaN,26.93,18.66,6.01,False,wet
1,2019-01-02,5.956522,27.25,24.26,7.08,False,wet
2,2019-01-03,7.260870,27.56,5.69,6.36,False,wet
3,2019-01-04,14.041667,27.83,0.41,4.82,False,wet
4,2019-01-05,26.083333,27.86,1.87,3.96,False,wet
5,2019-01-06,39.333333,27.98,0.97,3.27,False,wet
6,2019-01-07,31.916667,28.21,1.78,3.53,False,wet
7,2019-01-08,25.791667,27.92,42.60,2.17,False,wet
8,2019-01-09,29.166667,27.60,7.02,3.76,False,wet
9,2019-01-10,20.291667,27.39,4.42,2.45,False,wet
